# CSCI203 Assignment 1 — Stage 2: 2-Teller Bank Simulation

This notebook extends the one-teller version into a **2-teller discrete-event simulation**.

The structure follows the one-teller notebook:

- same input reading method,
- same customer representation,
- same manually implemented priority queue,
- same statistics collection,
- additional teller management using Round Robin.

New rule added:

- When a customer arrives and multiple tellers are available, the next idle teller is selected using Round Robin.


## 2. Imports

I only used `sys` and `numpy` following the assignment instructions.


In [1]:
import sys
import numpy


## 3. Customer-record positions

Each customer is stored as:

`[arrival_time, service_time, priority, original_order]`

The constants below make the code easier to understand.


In [2]:
ARRIVAL = 0
SERVICE = 1
PRIORITY = 2
ORDER = 3


## 4. Read the input file

The file is read record by record. The last `0 0` record is a dummy record and is **not** stored as a customer.


In [3]:
def read_customers(filename):
    customers = []
    sequence_number = 0
    previous_arrival = -1.0

    try:
        input_file = open(filename, "r")
    except OSError:
        print("ERROR: Could not open file:", filename)
        sys.exit(1)

    for raw_line in input_file:
        line = raw_line.strip()

        # Ignore completely blank lines.
        if line == "":
            continue

        parts = line.split()

        # Every useful line must contain at least arrival and service time.
        if len(parts) < 2:
            input_file.close()
            raise ValueError("Invalid input line: " + line)

        arrival_time = float(parts[0])
        service_time = float(parts[1])

        # The dummy record may contain only the two zeros.
        if arrival_time == 0.0 and service_time == 0.0:
            break

        if len(parts) < 3:
            input_file.close()
            raise ValueError("Customer record is missing its priority: " + line)

        priority = int(parts[2])

        if priority < 1 or priority > 3:
            input_file.close()
            raise ValueError("Priority must be 1, 2, or 3: " + line)

        # The assignment says records are already sorted by arrival time.
        if arrival_time < previous_arrival:
            input_file.close()
            raise ValueError("Input records are not in ascending arrival-time order.")

        customer = [arrival_time, service_time, priority, sequence_number]
        customers.append(customer)

        previous_arrival = arrival_time
        sequence_number += 1

    input_file.close()
    return customers


## 5. Implement the priority queue ourselves

The queue is a normal list, but i have manually inserted each new waiting customer in the correct position.

Ordering rule:

1. higher priority first;
2. if priorities are equal, earlier arrival first;
3. if arrival and priority are both equal, preserve original file order.


In [4]:
def customer_should_come_before(customer_a, customer_b):
    # Higher priority comes first.
    if customer_a[PRIORITY] > customer_b[PRIORITY]:
        return True
    if customer_a[PRIORITY] < customer_b[PRIORITY]:
        return False

    # Same priority: earlier arrival comes first.
    if customer_a[ARRIVAL] < customer_b[ARRIVAL]:
        return True
    if customer_a[ARRIVAL] > customer_b[ARRIVAL]:
        return False

    # Exact tie: preserve original input order.
    return customer_a[ORDER] < customer_b[ORDER]


def priority_enqueue(queue, customer):
    # Find where the customer belongs.
    insert_position = len(queue)

    for i in range(len(queue)):
        if customer_should_come_before(customer, queue[i]):
            insert_position = i
            break

    # Make one extra space at the end.
    queue.append(customer)

    # Shift existing elements one position to the right.
    i = len(queue) - 1
    while i > insert_position:
        queue[i] = queue[i - 1]
        i -= 1

    queue[insert_position] = customer


def priority_dequeue(queue):
    # The first element is always the next customer to serve.
    first_customer = queue[0]

    # Shift remaining customers one place to the left.
    for i in range(1, len(queue)):
        queue[i - 1] = queue[i]

    queue.pop()
    return first_customer


## 6. 2-Teller discrete-event simulation

The simulation clock jumps directly between important events:

1. Customer arrival
2. Teller service completion

Each teller stores:

- current status (busy/idle),
- finishing time,
- number of customers served,
- total busy time.

For selecting an available teller:

- Round Robin starts searching from the next teller position.
- After assigning a customer, the pointer moves forward.


In [5]:

def find_idle_teller(tellers, start_position):
    # Search all tellers in circular Round Robin order.
    number_of_tellers = len(tellers)

    for i in range(number_of_tellers):
        teller_number = (start_position + i) % number_of_tellers

        # Return the first available teller.
        if tellers[teller_number]["busy"] == False:
            return teller_number

    # No teller is available.
    return -1


def simulate_multi_teller(customers):

    # Create teller information.
    tellers = []

    for i in range(2):
        tellers.append({
            "busy": False,
            "finish_time": float("inf"),
            "served": 0,
            "busy_time": 0.0
        })

    waiting_queue = []

    current_time = 0.0
    customer_index = 0
    round_robin_pointer = 0

    total_waiting_time = 0.0
    total_service_time = 0.0
    maximum_queue_length = 0

    total_customers = len(customers)


    while (customer_index < total_customers or
           len(waiting_queue) > 0 or
           any(t["busy"] for t in tellers)):


        # Find next customer arrival.
        if customer_index < total_customers:
            next_arrival = customers[customer_index][ARRIVAL]
        else:
            next_arrival = float("inf")


        # Find earliest teller completion.
        next_completion = float("inf")
        completed_teller = -1

        for i in range(len(tellers)):
            if tellers[i]["busy"]:
                if tellers[i]["finish_time"] < next_completion:
                    next_completion = tellers[i]["finish_time"]
                    completed_teller = i


        # Event 1: Teller completes service.
        if next_completion <= next_arrival:

            current_time = next_completion

            teller = tellers[completed_teller]

            # Teller becomes available.
            teller["busy"] = False
            teller["finish_time"] = float("inf")


            # If customers are waiting, immediately serve next customer.
            if len(waiting_queue) > 0:

                customer = priority_dequeue(waiting_queue)

                waiting_time = current_time - customer[ARRIVAL]

                total_waiting_time += waiting_time
                total_service_time += customer[SERVICE]

                teller["served"] += 1
                teller["busy_time"] += customer[SERVICE]

                teller["busy"] = True
                teller["finish_time"] = current_time + customer[SERVICE]


        # Event 2: Customer arrives.
        else:

            current_time = next_arrival

            customer = customers[customer_index]

            customer_index += 1


            # Search for an idle teller using Round Robin.
            teller_number = find_idle_teller(
                tellers,
                round_robin_pointer
            )


            if teller_number != -1:

                teller = tellers[teller_number]

                teller["served"] += 1
                teller["busy_time"] += customer[SERVICE]

                total_service_time += customer[SERVICE]

                teller["busy"] = True
                teller["finish_time"] = current_time + customer[SERVICE]


                # Move Round Robin pointer.
                round_robin_pointer = (
                    teller_number + 1
                ) % len(tellers)


            else:

                # All tellers busy.
                # Put customer into priority queue.
                priority_enqueue(
                    waiting_queue,
                    customer
                )


                if len(waiting_queue) > maximum_queue_length:
                    maximum_queue_length = len(waiting_queue)



    results = {

        "served_each_teller":
            [t["served"] for t in tellers],

        "total_time":
            current_time,

        "average_service_time":
            total_service_time / total_customers,

        "average_waiting_time":
            total_waiting_time / total_customers,

        "maximum_queue_length":
            maximum_queue_length,

        "average_queue_length":
            total_waiting_time / current_time,

        "idle_rates":
            []
    }


    # Calculate idle rate for every teller.
    for teller in tellers:

        idle_time = current_time - teller["busy_time"]

        results["idle_rates"].append(
            idle_time / current_time
        )


    return results


## 7. Print required statistics

This output format is prepared for the final report.


In [6]:
def print_statistics(results):
    print("\n================ MULTI-TELLER RESULTS ================")
    print(
        "Number of tellers used                 :",
        len(results["served_each_teller"])
    )

    # Number of customers served by each teller
    for i in range(len(results["served_each_teller"])):
        print(
            "Number of customers served by Teller {} : {}".format(
                i,
                results["served_each_teller"][i]
            )
        )

    # Overall simulation statistics
    print(
        "Total simulation time                  : {:.4f}".format(
            results["total_time"]
        )
    )
    print(
        "Average service time per customer      : {:.4f}".format(
            results["average_service_time"]
        )
    )
    print(
        "Average waiting time per customer      : {:.4f}".format(
            results["average_waiting_time"]
        )
    )
    print(
        "Maximum length of queue                : {}".format(
            results["maximum_queue_length"]
        )
    )
    print(
        "Average length of queue                : {:.4f}".format(
            results["average_queue_length"]
        )
    )
    # Idle rate of each teller
    print("\n---------------- TELLER IDLE RATES -------------------")
    for i in range(len(results["idle_rates"])):
        print(
            "Idle rate of Teller {}                  : {:.4f} ({:.2f}%)".format(
                i,
                results["idle_rates"][i],
                results["idle_rates"][i] * 100
            )
        )
    print("\n======================================================")

## 8. Load input file


In [7]:
# For two-teller simulation

number_of_tellers = 2

# User provides only the input file name
filename = input("Enter input filename for two tellers: ").strip()

customers = read_customers(filename)

print("Loaded", len(customers), "customers from", filename)
print("Number of tellers:", number_of_tellers)

Loaded 100 customers from a1-sample.txt
Number of tellers: 2


## 9. Run simulation and record runtime


In [8]:

%%time

results = simulate_multi_teller(customers)

print_statistics(results)



================ MULTI-TELLER RESULTS ================
Number of tellers used                 : 2
Number of customers served by Teller 0 : 49
Number of customers served by Teller 1 : 51
Total simulation time                  : 653.3107
Average service time per customer      : 12.7983
Average waiting time per customer      : 60.3841
Maximum length of queue                : 22
Average length of queue                : 9.2428

---------------- TELLER IDLE RATES -------------------
Idle rate of Teller 0                  : 0.0299 (2.99%)
Idle rate of Teller 1                  : 0.0111 (1.11%)

CPU times: total: 15.6 ms
Wall time: 2.03 ms
